In [ ]:
from dotenv import load_dotenv
import os
import re
import unicodedata
import pandas as pd
from sqlalchemy import create_engine, text

load_dotenv()

user = os.getenv("SUPABASE_DB_USER")
pwd = os.getenv("SUPABASE_DB_PASSWORD")
host = os.getenv("SUPABASE_DB_HOST")
port = os.getenv("SUPABASE_DB_PORT", "5432")
db = os.getenv("SUPABASE_DB_NAME")

engine = create_engine(f"postgresql+psycopg2://{user}:{pwd}@{host}:{port}/{db}")


def normalize_legacy_text(text: str) -> str:
    text = str(text or "").replace("ç", "c").replace("Ç", "C")
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    return text


def sanitize_table_name(name: str) -> str:
    s = normalize_legacy_text(name).strip().lower()
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^a-z0-9_]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    if not s:
        s = "tabela"
    if not re.match(r"^[a-z_]", s):
        s = "_" + s
    return s[:63]


# coloque aqui a tabela selecionada no dropdown
selected_table = "Recursos Operacionais"


q_recursos = text("""
select
    id_recurso,
    recurso_name,
    recurso_description
from public."antt_antt_recursos_dim"
""")

df_recursos = pd.read_sql_query(q_recursos, engine)
df_recursos["recurso_name_tratado"] = df_recursos["recurso_name"].apply(sanitize_table_name)

selected_table_tratada = sanitize_table_name(selected_table)

match = df_recursos[df_recursos["recurso_name_tratado"] == selected_table_tratada].copy()

print("\nTabela selecionada:")
print(selected_table)

print("\nTabela selecionada tratada:")
print(selected_table_tratada)

if match.empty:
    print("\nNenhum recurso encontrado com esse nome tratado.")
else:
    print("\nRecurso encontrado:")
    print(match[["id_recurso", "recurso_name", "recurso_name_tratado", "recurso_description"]].to_string(index=False))

    id_recurso = match.iloc[0]["id_recurso"]
    recurso_description = match.iloc[0]["recurso_description"]

    print("\nid_recurso encontrado:")
    print(id_recurso)

    print("\nDescrição encontrada:")
    print(recurso_description)

    q_dicionario = text("""
    select *
    from public."antt_antt_dicionarios_campos"
    where id_recurso = :id_recurso
    """)

    df_dicionario = pd.read_sql_query(q_dicionario, engine, params={"id_recurso": id_recurso})

    print("\nDicionário filtrado por resource_id:")
    if df_dicionario.empty:
        print("Nenhum registro encontrado.")
    else:
        print(df_dicionario.to_string(index=False))


Tabela selecionada:
Município

Tabela selecionada tratada:
municipio

Recurso encontrado:
                      id_recurso recurso_name recurso_name_tratado                                                 recurso_description
27884866c734312774ca00c2f7c56f16    Município            municipio Informações sobre os municipios que as rodovias concediadas cruzam.

id_recurso encontrado:
27884866c734312774ca00c2f7c56f16

Descrição encontrada:
Informações sobre os municipios que as rodovias concediadas cruzam.

Dicionário filtrado por resource_id:
            Campo           Tipo    Formato                                                                                                                           Descricao                       id_dataset                       id_recurso                          resource_id                                          dataset_name                                                                                                                         